In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install transformers datasets pandas torch sentencepiece -q

In [3]:
import pandas as pd

file_path = "/content/drive/MyDrive/cloth/amazon-fashion-800k+-user-reviews-dataset.csv"

df = pd.read_csv(file_path)
df.head()
print(df.shape)
print(df['target'].value_counts())
print(df[['text', 'target']].head())

(867310, 11)
target
-1    346924
 1    346924
 0    173462
Name: count, dtype: int64
                                                text  target
0  I was looking for 5 pair and only received 2 p...      -1
1  Just donÃ¢ÂÂt. These things fell apart after...      -1
2                        Retuned is too small for me      -1
3  This product came with the sleeves turned insi...      -1
4  Worn once and several places at seams have com...      -1


In [4]:
import re

# Keep only needed columns
df = df[['text', 'target']].dropna()

# Clean text
def clean_text(text):
    text = str(text)
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)   # remove non-ASCII (fixes DonÃ¢ etc)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text'] = df['text'].apply(clean_text)

# Remove very short reviews
df = df[df['text'].str.len() > 20]

# Map target to sentiment label
sentiment_map = {1: 'positive', 0: 'neutral', -1: 'negative'}
df['sentiment'] = df['target'].map(sentiment_map)

print(df['sentiment'].value_counts())
print(f"Total usable reviews: {len(df)}")

sentiment
negative    320481
positive    301248
neutral     161322
Name: count, dtype: int64
Total usable reviews: 783051


In [5]:
# Clothing-specific aspect keywords
ASPECT_KEYWORDS = {
    'fabric':    ['fabric', 'material', 'cloth', 'cotton', 'silk', 'polyester', 'linen', 'texture', 'soft', 'rough', 'thin', 'thick'],
    'fit':       ['fit', 'fitted', 'fitting', 'loose', 'tight', 'snug', 'baggy', 'oversized'],
    'size':      ['size', 'sizing', 'small', 'large', 'medium', 'runs small', 'runs large', 'true to size'],
    'stitching': ['stitch', 'stitching', 'seam', 'seams', 'sewing', 'thread', 'hem', 'hemline'],
    'color':     ['color', 'colour', 'faded', 'vibrant', 'bright', 'dark', 'dye', 'print', 'pattern'],
    'design':    ['design', 'style', 'look', 'appearance', 'aesthetic', 'beautiful', 'ugly', 'elegant', 'pretty'],
    'quality':   ['quality', 'durable', 'durability', 'cheap', 'premium', 'well made', 'poorly made'],
    'delivery':  ['delivery', 'shipping', 'arrived', 'package', 'packaging', 'shipped'],
    'price':     ['price', 'cost', 'worth', 'expensive', 'cheap', 'affordable', 'value'],
    'zipper':    ['zipper', 'zip', 'button', 'buttons', 'closure', 'snap'],
    'comfort':   ['comfort', 'comfortable', 'uncomfortable', 'cozy', 'itchy', 'breathable'],
    'length':    ['length', 'long', 'short', 'maxi', 'mini', 'crop', 'ankle'],
}

def extract_aspects_from_text(text: str, sentiment: str) -> str:
    """
    Finds which aspects are mentioned in the review
    and assigns the overall sentiment to each found aspect.
    Returns: 'fabric: positive, size: negative' etc.
    """
    text_lower = text.lower()
    found_aspects = []

    for aspect, keywords in ASPECT_KEYWORDS.items():
        for keyword in keywords:
            if keyword in text_lower:
                found_aspects.append(aspect)
                break  # only add each aspect once

    if not found_aspects:
        return None  # skip reviews with no detectable aspects

    # Build output string
    pairs = [f"{aspect}: {sentiment}" for aspect in found_aspects]
    return ", ".join(pairs)

# Apply to dataset
df['output'] = df.apply(
    lambda row: extract_aspects_from_text(row['text'], row['sentiment']),
    axis=1
)

# Drop rows where no aspects were found
df = df.dropna(subset=['output'])

print(f"Reviews with aspects found: {len(df)}")
print("\nSample outputs:")
print(df[['text', 'output']].head(10).to_string())

Reviews with aspects found: 660385

Sample outputs:
                                                                                                                                                                                                                                        text                                                             output
0   I was looking for 5 pair and only received 2 pair tho I paid for 5 pair and the two pair was late I'm a prime member and feel like I was cheated if my math serve me correct I'm still short 3 pair of panties that were listed as prime                                 design: negative, length: negative
1                                                                                                                            Just don t. These things fell apart after the first wash and are SO uncomfortable. Buy literally anything else!                                fabric: negative, comfort: negative
2                                   

In [6]:
# Take 100k samples, balanced across sentiments
sample_per_class = 33_333

df_balanced = pd.concat([
    df[df['sentiment'] == 'positive'].sample(min(sample_per_class, len(df[df['sentiment'] == 'positive'])), random_state=42),
    df[df['sentiment'] == 'negative'].sample(min(sample_per_class, len(df[df['sentiment'] == 'negative'])), random_state=42),
    df[df['sentiment'] == 'neutral'].sample(min(sample_per_class,  len(df[df['sentiment'] == 'neutral'])),  random_state=42),
]).sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

print(f"Final training size: {len(df_balanced)}")
print(df_balanced['sentiment'].value_counts())

Final training size: 99999
sentiment
positive    33333
neutral     33333
negative    33333
Name: count, dtype: int64


In [7]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "t5-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 64

def preprocess(examples):
    inputs  = [f"aspect-sentiment analysis: {text}" for text in examples["text"]]
    targets = examples["output"]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        padding="max_length",
        truncation=True
    )
    labels = tokenizer(
        targets,
        max_length=MAX_TARGET_LENGTH,
        padding="max_length",
        truncation=True
    )

    # Replace pad token with -100 so loss ignores padding
    model_inputs["labels"] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in label]
        for label in labels["input_ids"]
    ]
    return model_inputs

# Convert to HuggingFace Dataset
dataset = Dataset.from_pandas(df_balanced[['text', 'output']])
dataset = dataset.map(preprocess, batched=True, remove_columns=["text", "output"])
dataset = dataset.train_test_split(test_size=0.20, seed=42)

print(f"Train: {len(dataset['train'])}, Val: {len(dataset['test'])}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Map:   0%|          | 0/99999 [00:00<?, ? examples/s]

Train: 79999, Val: 20000


In [ ]:
from transformers import (
    T5ForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
import torch

model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

training_args = Seq2SeqTrainingArguments(
    output_dir="./t5base_clothing",
    num_train_epochs=5,                    # 5 is enough for this dataset size
    per_device_train_batch_size=32,        # increase to 32 if Colab allows
    per_device_eval_batch_size=16,
    learning_rate=3e-4,
    warmup_steps=100,
    weight_decay=0.01,
    predict_with_generate=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),        # faster training on GPU
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
)

print("Starting training...")
trainer.train()
print("Training complete!")

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Starting training...


Epoch,Training Loss,Validation Loss


In [ ]:
model.eval()

test_reviews = [
    "the fabric is very soft but the size runs too small",
    "zipper broke after one use, terrible quality",
    "beautiful design and perfect fit, fast delivery",
    "color faded after first wash and stitching came apart",
    "comfortable material but a bit overpriced",
]

print("Testing model predictions:\n")
for review in test_reviews:
    inputs = tokenizer(
        f"aspect-sentiment analysis: {review}",
        return_tensors="pt",
        max_length=128,
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=4,
            do_sample=False,
            no_repeat_ngram_size=3,
            repetition_penalty=2.5,
            early_stopping=True
        )

    result = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"Review : {review}")
    print(f"Output : {result}")
    print("-" * 60)

In [ ]:
# ─── FULL MODEL TEST ────────────────────────────────────────
import torch

model.eval()

test_reviews = [
    # Single aspect tests
    "the fabric is extremely soft and comfortable",
    "size runs very small, had to return it",
    "color faded completely after first wash",
    "zipper broke after wearing it once",
    "delivery was super fast and well packaged",

    # Multi aspect tests (most important)
    "the fabric is soft but the size runs too small",
    "beautiful design but zipper broke after one use",
    "great fit and color but stitching came apart",
    "fast delivery but color looks nothing like the photo",
    "comfortable material and perfect fit but overpriced",

    # Real-style amazon reviews
    "Very nice and elegant dress, fits perfectly and fabric feels premium",
    "Returned it immediately, stitching was poor and size was way off",
    "Love the color and design but it runs small, order a size up",
    "Cheap material, fell apart after one wash, not worth the money",
    "Perfect for the price, good quality fabric and fast shipping",
]

print("=" * 65)
print("MODEL TEST RESULTS")
print("=" * 65)

passed = 0
for i, review in enumerate(test_reviews, 1):
    inputs = tokenizer(
        f"aspect-sentiment analysis: {review}",
        return_tensors="pt",
        max_length=128,
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=4,
            do_sample=False,
            no_repeat_ngram_size=3,
            repetition_penalty=2.5,
            early_stopping=True
        )

    result = tokenizer.decode(output[0], skip_special_tokens=True)

    # Check if output looks valid (has "aspect: sentiment" format)
    is_valid = ":" in result and any(
        word in result for word in
        ['positive', 'negative', 'neutral']
    )
    status = "✅" if is_valid else "❌"
    if is_valid:
        passed += 1

    print(f"\n[{i:02d}] {status} Review : {review[:60]}...")
    print(f"        Output : {result}")

print("\n" + "=" * 65)
print(f"SCORE: {passed}/{len(test_reviews)} reviews produced valid output")

if passed == len(test_reviews):
    print("✅ PERFECT - Model is working correctly! Safe to download.")
elif passed >= len(test_reviews) * 0.8:
    print("✅ GOOD - Model is working well. Safe to download.")
elif passed >= len(test_reviews) * 0.5:
    print("⚠️  AVERAGE - Model works but could be better.")
    print("   Consider training 2-3 more epochs.")
else:
    print("❌ POOR - Model needs more training.")
    print("   Run trainer.train() again for 3 more epochs.")

In [ ]:
import shutil

# Save model
model.save_pretrained("./t5base_clothing")
tokenizer.save_pretrained("./t5base_clothing")

# Zip and download
shutil.make_archive("t5base", "zip", "./t5base_clothing")
files.download("t5base.zip")

print("Downloaded! Unzip and replace your existing t5base folder.")

In [ ]:
# ─── SANITY CHECK - Run this before full training ───────────
import torch

print("=" * 60)
print("SANITY CHECK")
print("=" * 60)

# 1. Check GPU
print(f"\n✅ Device: {'GPU - ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌ CPU only - switch to GPU in Runtime settings'}")

# 2. Check dataset
print(f"\n✅ Train samples : {len(dataset['train'])}")
print(f"✅ Val samples   : {len(dataset['test'])}")

# 3. Check a raw sample
raw_sample = df_balanced.iloc[0]
print(f"\n✅ Sample review : {raw_sample['text'][:80]}...")
print(f"✅ Sample output : {raw_sample['output']}")

# 4. Check tokenized sample
tok_sample = dataset['train'][0]
print(f"\n✅ Input IDs length  : {len(tok_sample['input_ids'])}")
print(f"✅ Labels length     : {len(tok_sample['labels'])}")
print(f"✅ Non-padding labels: {sum(1 for x in tok_sample['labels'] if x != -100)}")

# 5. Quick model forward pass (checks model loads and runs without error)
print("\n⏳ Testing model forward pass...")
model.eval()
test_input = tokenizer(
    "aspect-sentiment analysis: the fabric is soft but size runs small",
    return_tensors="pt",
    max_length=128,
    truncation=True
).to(model.device)

with torch.no_grad():
    output = model.generate(
        **test_input,
        max_new_tokens=64,
        num_beams=4,
        do_sample=False,
        no_repeat_ngram_size=3,
        repetition_penalty=2.5,
        early_stopping=True
    )

result = tokenizer.decode(output[0], skip_special_tokens=True)
print(f"✅ Model output (before training): '{result}'")
print("   ^ This will be garbage before training - that's normal!")

# 6. Check trainer is ready
print(f"\n✅ Trainer ready    : {trainer is not None}")
print(f"✅ Epochs planned   : {training_args.num_train_epochs}")
print(f"✅ Batch size       : {training_args.per_device_train_batch_size}")
print(f"✅ fp16 (GPU speed) : {training_args.fp16}")

print("\n" + "=" * 60)
print("ALL CHECKS PASSED - Safe to run trainer.train()" )
print("=" * 60)

In [ ]:
# ─── FULL MODEL TEST ────────────────────────────────────────
import torch

model.eval()

test_reviews = [
    # Single aspect tests
    "the fabric is extremely soft and comfortable",
    "size runs very small, had to return it",
    "color faded completely after first wash",
    "zipper broke after wearing it once",
    "delivery was super fast and well packaged",

    # Multi aspect tests (most important)
    "the fabric is soft but the size runs too small",
    "beautiful design but zipper broke after one use",
    "great fit and color but stitching came apart",
    "fast delivery but color looks nothing like the photo",
    "comfortable material and perfect fit but overpriced",

    # Real-style amazon reviews
    "Very nice and elegant dress, fits perfectly and fabric feels premium",
    "Returned it immediately, stitching was poor and size was way off",
    "Love the color and design but it runs small, order a size up",
    "Cheap material, fell apart after one wash, not worth the money",
    "Perfect for the price, good quality fabric and fast shipping",
]

print("=" * 65)
print("MODEL TEST RESULTS")
print("=" * 65)

passed = 0
for i, review in enumerate(test_reviews, 1):
    inputs = tokenizer(
        f"aspect-sentiment analysis: {review}",
        return_tensors="pt",
        max_length=128,
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=4,
            do_sample=False,
            no_repeat_ngram_size=3,
            repetition_penalty=2.5,
            early_stopping=True
        )

    result = tokenizer.decode(output[0], skip_special_tokens=True)

    # Check if output looks valid (has "aspect: sentiment" format)
    is_valid = ":" in result and any(
        word in result for word in
        ['positive', 'negative', 'neutral']
    )
    status = "✅" if is_valid else "❌"
    if is_valid:
        passed += 1

    print(f"\n[{i:02d}] {status} Review : {review[:60]}...")
    print(f"        Output : {result}")

print("\n" + "=" * 65)
print(f"SCORE: {passed}/{len(test_reviews)} reviews produced valid output")

if passed == len(test_reviews):
    print("✅ PERFECT - Model is working correctly! Safe to download.")
elif passed >= len(test_reviews) * 0.8:
    print("✅ GOOD - Model is working well. Safe to download.")
elif passed >= len(test_reviews) * 0.5:
    print("⚠️  AVERAGE - Model works but could be better.")
    print("   Consider training 2-3 more epochs.")
else:
    print("❌ POOR - Model needs more training.")
    print("   Run trainer.train() again for 3 more epochs.")